# DCT Laboratory — Volume I, Chapter 8
## Stochastic Enterprise Dynamics
**Seed `26108`** · Companion to Chapter 8 and **AXIOM Lab 1.8** (module **AXIOM-08**) · Mirrored in `DCT_V1_Ch08_Lab.xlsx`

This laboratory reproduces **Worked Example 8.1, *Will the Digital Program Breach the Floor?***, with the book's
declarations, and works the five steps of AXIOM Lab 1.8, *Putting Odds on Meridian's Plan*: watch the fan form,
check the multi-year view, price an event risk, feel the flaw of averages, and put odds on the program. The
worked example's 100,000 paths are drawn, as in the book, with **random seed 8** (Volume II, Online Appendix
1.B.4), so its odds come out exactly as printed; the lab's seed 26108 is used only for fresh draws, labelled as
such. Every Meridian number below is the book's or follows from its declarations (Worked Example 8.1, Table 8.5,
Figures 8.1–8.4, Sections 8.6–8.9, Online Appendix 8.B); the final cell checks the printed ones. Money is in
\$ million.

## The declarations (Worked Example 8.1, Step 1; Sections 8.6 and 8.7)

**The plan.** The digital program's liquidity path of Chapters 1 and 5, read quarterly over the program's three
years: Chapter 1's liquidity index (today 100, that is \$620 million, \$6.2 million a point) relaxes toward 112
at 35 percent a year, less the program's cash draw, which deepens to 28 points after two years. Liquidity dips
from \$620 million to a trough of \$482 million in quarter 7 and recovers to \$510 million by the end of year 3,
\$132 million above the board's floor of \$350 million (Table 5.4). The board's risk appetite tolerates at most a
5 percent chance of breaching the floor during the program.

**The uncertainty** (Step 1). *Execution and market deviation:* each quarter brings a surprise with a standard
deviation of \$50 million, and 60 percent of a surprise carries into the next quarter, $E_{k+1} = 0.6E_k + w_k$.
*Funding-window events:* a window closes about once a decade ($\lambda = 0.10$ a year), and each closure removes
on average \$120 million of liquidity (standard deviation \$50 million; gamma distributed in the book's engine)
that is not recovered within the program. Liquidity is the plan plus the deviation less the closures. *The
levers* (Step 5): a standby line that replaces up to \$100 million of any closure, and a phased build whose
cash draw deepens to 20 points instead of 28.

**Two more Meridian declarations.** Industrial Systems' quarterly revenue deviation from plan follows
$R_{k+1} = 0.7R_k + w_k$ with shocks of standard deviation \$15 million, and a covenant test bites at \$45 million
below plan (Section 8.6). Regulatory interventions arrive at $\lambda = 0.15$ a year and cost on average
\$40 million, with a standard deviation of \$30 million (Section 8.7).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, exp
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26108          # the lab's seed: fresh draws only, labelled as such
BOOK_SEED = 8         # the book's stream for Worked Example 8.1 (Volume II, Online Appendix 1.B.4)

def Phi(z):
    return 0.5 * (1 + erf(z / sqrt(2)))

# Worked Example 8.1 ($ million)
USD_PER_POINT, QUARTERS, N_PATHS = 6.2, 12, 100_000
FLOOR, APPETITE = 350.0, 0.05
A_E, SD_W = 0.6, 50.0                                   # deviation: share carried over; surprise sd a quarter
LAM, J_MEAN, J_SD = 0.10, 120.0, 50.0                   # closures a year; size: mean and sd
SHAPE, SCALE = (J_MEAN / J_SD) ** 2, J_SD ** 2 / J_MEAN # gamma sizes with that mean and sd
LINE, DEPTH, DEPTH_PHASED = 100.0, 28.0, 20.0           # Step 5's levers; the plan's draw depth

def plan(depth=DEPTH):
    # Chapter 1's digital liquidity index, quarterly for three years, in $ million
    t = np.arange(QUARTERS + 1) / 4
    return USD_PER_POINT * (112 + (100 - 112) * np.exp(-0.35 * t) - depth * (t / 2) * np.exp(1 - t / 2))

def simulate(noise=True, events=True, line=0.0, depth=DEPTH, n=N_PATHS, seed=BOOK_SEED):
    # Liquidity paths [n, 13] and each path's number of closures. The draws follow the book's engine,
    # in its order: each quarter the n surprises, the n closure counts, then the sizes of that quarter's
    # closures in path order. A channel switched off keeps its draws, so every scenario sees the same
    # futures; seed 8 gives the book's.
    rng = np.random.default_rng(seed)
    pl, E, J = plan(depth), np.zeros(n), np.zeros(n)
    L = np.empty((n, QUARTERS + 1)); L[:, 0] = pl[0]
    closures = np.zeros(n, dtype=int)
    for k in range(QUARTERS):
        w = SD_W * rng.standard_normal(n)
        cnt = rng.poisson(LAM / 4, n)
        sizes = rng.gamma(SHAPE, SCALE, cnt.sum())
        lost = np.bincount(np.repeat(np.arange(n), cnt), np.maximum(sizes - line, 0.0), minlength=n)
        if noise:
            E = A_E * E + w
        if events:
            J, closures = J - lost, closures + cnt
        L[:, k + 1] = pl[k + 1] + E + J
    return L, closures

def odds(L):
    # breach probability, its standard error, and each path's lowest quarter
    lows = L[:, 1:].min(axis=1)
    p = float((lows < FLOOR).mean())
    return p, sqrt(p * (1 - p) / len(lows)), lows

# Section 8.6: Industrial Systems' revenue deviation; Section 8.7: regulatory interventions ($ million)
A_R, SD_R, R0, COVENANT = 0.7, 15.0, 30.0, 45.0
LAM_R, JR_MEAN, JR_SD = 0.15, 40.0, 30.0

pl = plan()
print("plan by quarter:", np.round(pl).astype(int))
print(f"trough ${pl.min():.0f} million in quarter {pl.argmin()}, ${pl.min() - FLOOR:.0f} million above the floor; "
      f"${pl[-1]:.0f} million at the end of year 3")
print(f"closure sizes: gamma, shape {SHAPE:.2f} and scale {SCALE:.2f} (mean {SHAPE * SCALE:.0f}, sd {sqrt(SHAPE) * SCALE:.0f})")

## Panel 1 — Watch the fan form (Theorems 8.2 and 8.3; Figures 8.1–8.2: a budget of \$21 million, 1.6 percent)

AXIOM Lab 1.8, step 1. Start Industrial Systems' revenue deviation after a strong quarter, $R_0 = 30$. Its law
stays normal, and its mean and variance follow the exact recursions of Theorem 8.3(i), $m_{k+1} = 0.7\,m_k$ and
$\sigma^2_{k+1} = 0.49\,\sigma^2_k + 15^2$: the fan drifts back toward plan and widens (means 21, 15 and 7,
standard deviations 15, 18 and 20 after one, two and four quarters; Figure 8.1) until it settles at the long-run
variance $15^2/(1 - 0.7^2) \approx 441$. The **volatility budget is \$21 million, not \$15 million**:
persistence inflates the naive figure by $1/\sqrt{1 - 0.49} = 1.40$. A covenant test that bites at \$45 million
below plan fails with a long-run chance of $\Phi(-45/21) \approx$ **1.6 percent** a quarter, against 0.13 percent
if persistence were ignored. Sweep the persistence through one and the band stops settling: with no restoring
force the spread grows like $15\sqrt{k}$ (Figure 8.2). Halving the persistence to 0.35 cuts the budget to
\$16.0 million and the chance to 0.25 percent (Online Appendix 8.B.4); Online Appendix 8.B.3 solves the same
Lyapunov equation for two coupled readings. Fresh paths drawn with the lab's seed scatter while their law moves
by the exact rule.

In [ ]:
def revenue_law(k, a=A_R, s=SD_R, r0=R0):
    # Theorem 8.3(i): R_k is normal, mean a^k r0, variance s^2 (1 + a^2 + ... + a^(2k-2))
    var = s ** 2 * k if a == 1 else s ** 2 * (1 - a ** (2 * k)) / (1 - a ** 2)
    return a ** k * r0, sqrt(var)

VAR_INF = SD_R ** 2 / (1 - A_R ** 2)
BUDGET = sqrt(VAR_INF)
P_COV, P_NAIVE = Phi(-COVENANT / BUDGET), Phi(-COVENANT / SD_R)
rng = np.random.default_rng(SEED)                      # fresh draws, lab seed
R = np.empty((100_000, 41)); R[:, 0] = R0
for k in range(40):
    R[:, k + 1] = A_R * R[:, k] + SD_R * rng.standard_normal(100_000)
print("after a strong quarter (R0 = +30): the exact law, and 100,000 fresh paths (seed 26108)")
print("quarters    mean      sd    simulated: mean      sd")
for k in (1, 2, 4, 8, 40):
    m_, s_ = revenue_law(k)
    print(f"{k:8d} {m_:7.2f} {s_:7.2f} {R[:, k].mean():18.2f} {R[:, k].std():7.2f}")
print(f"long run: variance 15^2/(1 - 0.7^2) = {VAR_INF:.1f}; volatility budget ${BUDGET:.2f} million, "
      f"{BUDGET / SD_R:.2f} times the shock")
tail = float((R[:, 40] < -COVENANT).mean())
print(f"covenant test at -45: long-run chance {100 * P_COV:.2f}% a quarter (fresh paths, quarter 40: {100 * tail:.2f}% "
      f"+/- {100 * sqrt(tail * (1 - tail) / 1e5):.2f}); ignoring persistence {100 * P_NAIVE:.3f}%")

print("\nthe variance saturator: persistence, long-run sd, and the covenant test's chance a quarter")
for a in (0.0, 0.35, 0.5, 0.7, 0.9, 0.95, 0.99):
    sd_ = SD_R / sqrt(1 - a ** 2)
    print(f"  a = {a:4.2f}: sd {sd_:6.1f}   chance {100 * Phi(-COVENANT / sd_):6.2f}%")
print("  a = 1.00: no long run; sd after 4, 8, 20 and 40 quarters (15 sqrt k):",
      ", ".join(f"{revenue_law(k, a=1.0)[1]:.1f}" for k in (4, 8, 20, 40)))
HALF = SD_R / sqrt(1 - 0.35 ** 2)
P_HALF = Phi(-COVENANT / HALF)
print(f"halving persistence to 0.35 (Online Appendix 8.B.4): budget ${HALF:.2f} million, chance {100 * P_HALF:.3f}% "
      f"a quarter, {P_COV / P_HALF:.1f} times smaller, with no change in the market")

# Online Appendix 8.B.3: two readings, Sigma = A Sigma A' + S, iterated from zero and in closed form
A2, S2 = np.array([[0.8, 0.2], [0.0, 0.6]]), np.diag([0.04, 0.01])
SIG = np.zeros((2, 2))
for _ in range(400):
    SIG = A2 @ SIG @ A2.T + S2
s22 = 0.01 / (1 - 0.6 ** 2); s12 = 0.12 * s22 / 0.52; s11 = (0.32 * s12 + 0.04 * s22 + 0.04) / 0.36
print(f"two readings: Sigma_inf = [[{SIG[0, 0]:.5f}, {SIG[0, 1]:.6f}], [{SIG[1, 0]:.6f}, {SIG[1, 1]:.6f}]] "
      f"(closed form {s11:.5f}, {s12:.6f}, {s22:.6f}); long-run sds {sqrt(SIG[0, 0]):.3f} and {sqrt(SIG[1, 1]):.3f}")

fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6))
q = np.arange(41)
law = np.array([revenue_law(k) for k in q])
for i in range(25):
    ax.plot(q[:21], R[i, :21], color="#999999", lw=0.5, alpha=0.6)
ax.fill_between(q[:21], law[:21, 0] - 2 * law[:21, 1], law[:21, 0] + 2 * law[:21, 1], color="#1F4E79", alpha=0.15,
                lw=0, label="mean ± 2 sd, exact law")
ax.plot(q[:21], law[:21, 0], color="#1F4E79", lw=2, label=r"mean: $30 \times 0.7^{k}$")
ax.axhline(-COVENANT, color="#B71C1C", lw=0.9, ls="--", label="covenant test: 45 below plan")
ax.set(xlabel="quarters after a strong quarter", ylabel="revenue deviation from plan (\\$ million)", xlim=(0, 20),
       ylim=(-90, 130), xticks=range(0, 21, 4), title="Paths scatter; the law moves exactly")
ax.legend(frameon=False, fontsize=7, loc="upper right")
for a, col, ls in ((0.35, "#616161", ":"), (0.7, "#1F4E79", "-"), (0.9, "#E65100", "--"), (1.0, "#B71C1C", "-")):
    bx.plot(q, [2 * revenue_law(k, a=a)[1] for k in q], color=col, lw=1.8, ls=ls,
            label=f"a = {a:.2f}" + (": no restoring force" if a == 1 else f": settles at ±{2 * SD_R / sqrt(1 - a * a):.0f}"))
bx.set(xlabel="quarters", ylabel="half-width of the 2-sd band (\\$ million)", xlim=(0, 40), ylim=(0, 200),
       xticks=range(0, 41, 10), title="The saturator: sweep the persistence a")
bx.legend(frameon=False, fontsize=7, loc="upper left")
for z in (ax, bx):
    z.tick_params(labelsize=7); z.xaxis.label.set_size(8); z.yaxis.label.set_size(8); z.title.set_size(9)
fig.suptitle("Revenue's fan drifts back and settles, unless persistence reaches one (seed 26108)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 2 — Check the multi-year view (Theorem 8.2(ii); Online Appendix 8.B.2: 19.6 percent, not 30)

AXIOM Lab 1.8, step 2. Multi-year risk views must compose: the odds over $j + k$ periods pass through every
intermediate state after $j$ periods (the Chapman–Kolmogorov equation). The program office classifies each
quarter as on plan or off plan; from on plan the next quarter is on plan with probability 0.9, and from off plan
it returns to plan with probability 0.3. Three quarters ahead, starting on plan, the one-quarter view implies a
**19.6 percent** chance of being off plan, so a planning pack that showed 30 percent would be inconsistent with
its own one-quarter view. In the long run the program is off plan a quarter of the time, in spells of about 3.3
quarters. The revenue fan passes the same test: its one-year fan compounded three times is its three-year fan,
while a fan stretched by the square-root law ignores the restoring force.

In [ ]:
P = np.array([[0.9, 0.1], [0.3, 0.7]])                # a quarter: rows from on plan, from off plan
P2, P3 = P @ P, P @ P @ P
print("P^2 rows:", P2.round(4).tolist(), "  P^3 rows:", P3.round(4).tolist())
print("Chapman-Kolmogorov: P^2 P = P P^2 = P^3 ->", bool(np.allclose(P2 @ P, P3) and np.allclose(P @ P2, P3)))
OFF3 = P3[0, 1]
print(f"off plan three quarters after being on plan: {100 * OFF3:.1f}%; a pack showing 30% contradicts its own one-quarter view")
vals, vecs = np.linalg.eig(P.T)
PI = np.real(vecs[:, np.argmin(abs(vals - 1))]); PI = PI / PI.sum()
print(f"stationary: on plan {PI[0]:.2f}, off plan {PI[1]:.2f}; spells last {1 / P[0, 1]:.0f} quarters on plan, "
      f"{1 / P[1, 0]:.1f} off plan")
rng = np.random.default_rng(SEED)                      # fresh draws, lab seed: the checker by simulation
state = np.zeros(100_000, dtype=int)
for _ in range(3):
    state = (rng.random(100_000) < P[state, 1]).astype(int)
sim = float(state.mean())
print(f"100,000 simulated programs (seed 26108): {100 * sim:.2f}% off plan after three quarters, "
      f"+/- {100 * sqrt(sim * (1 - sim) / 1e5):.2f}")

a4, V1 = A_R ** 4, SD_R ** 2 * (1 - A_R ** 8) / (1 - A_R ** 2)       # revenue: the one-year kernel
V3_COMPOSED = V1 * (1 + a4 ** 2 + a4 ** 4)                          # three one-year fans compounded
V3_DIRECT = SD_R ** 2 * (1 - A_R ** 24) / (1 - A_R ** 2)            # the twelve-quarter fan
print(f"revenue fan: one-year sd {sqrt(V1):.2f}; three-year sd compounded {sqrt(V3_COMPOSED):.4f}, direct "
      f"{sqrt(V3_DIRECT):.4f}; the mean's decay 0.7^12 = {A_R ** 12:.4f} = (0.7^4)^3")
print(f"a three-year fan stretched by the square-root law, sqrt(3) x {sqrt(V1):.2f} = {sqrt(3 * V1):.1f}, "
      f"is not this model's: it ignores the restoring force")

## Panel 3 — Price an event risk (Theorem 8.4; Figure 8.3: \$6 million a year, \$19 and \$43 million, 14 percent)

AXIOM Lab 1.8, step 3. Meridian's regulatory interventions arrive on a Poisson clock at $\lambda = 0.15$ a year,
about once every 6.7 years, and cost on average \$40 million with a standard deviation of \$30 million, so
$\mathrm{E}[J^2] = 40^2 + 30^2 = 2{,}500$. Theorem 8.4(ii) gives event risk two ledger lines: the **plan line**
$\lambda\,\mathrm{E}[J] =$ \$6 million a year, which belongs in the expected plan, quiet years included, and the
**risk line** $\lambda\,\mathrm{E}[J^2] = 375$ a year, a standard deviation of about \$19 million over one year and
\$43 million over five. Halving the frequency while doubling the size leaves the plan line unchanged and doubles
the variance. And because the clock is memoryless (Theorem 8.4(i)), five quiet years leave next year's chance of
an intervention where it was, $1 - e^{-0.15} \approx 14$ percent. Fresh histories drawn with the lab's seed,
waiting times first and then sizes (Section 8.9), show all three.

In [ ]:
EJ2 = JR_MEAN ** 2 + JR_SD ** 2
PLAN_LINE, RISK_LINE = LAM_R * JR_MEAN, LAM_R * EJ2
SD1, SD5 = sqrt(RISK_LINE), sqrt(5 * RISK_LINE)
P_NEXT, P_FIVE = 1 - exp(-LAM_R), 1 - exp(-5 * LAM_R)
print(f"E[J^2] = 40^2 + 30^2 = {EJ2:,.0f}; plan line 0.15 x 40 = ${PLAN_LINE:.1f} million a year; "
      f"risk line 0.15 x 2,500 = {RISK_LINE:.0f} a year")
print(f"standard deviation ${SD1:.1f} million over one year, ${SD5:.1f} million over five; mean wait {1 / LAM_R:.1f} years")
print(f"half the frequency, twice the size: plan line {LAM_R / 2 * 2 * JR_MEAN:.1f}, variance {LAM_R / 2 * 4 * EJ2:.0f} a year")
print(f"chance of an intervention next year {100 * P_NEXT:.1f}%, after five quiet years as well; "
      f"at least one in five years {100 * P_FIVE:.1f}%")

rng = np.random.default_rng(SEED)                      # fresh draws, lab seed: 200,000 ten-year histories
N_H, T_H = 200_000, 10.0
times = np.cumsum(rng.exponential(1 / LAM_R, (N_H, 15)), axis=1)          # waiting times, then sizes
assert (times[:, -1] > T_H).all()
cost_sizes = rng.gamma((JR_MEAN / JR_SD) ** 2, JR_SD ** 2 / JR_MEAN, times.shape)
for t in (1.0, 5.0):
    c = np.where(times < t, cost_sizes, 0.0).sum(axis=1)
    print(f"  fresh histories, {t:.0f} year(s): mean {c.mean():.2f} (exact {PLAN_LINE * t:.0f}), "
          f"variance {c.var():.0f} (exact {RISK_LINE * t:.0f})")
quiet = times[:, 0] >= 5.0
in_year = lambda a, b: ((times >= a) & (times < b)).any(axis=1)
print(f"  memorylessness: an intervention in year 1, {100 * in_year(0, 1).mean():.1f}%; in year 6 after five quiet "
      f"years, {100 * in_year(5, 6)[quiet].mean():.1f}% ({quiet.sum():,} quiet histories)")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
t = np.linspace(0, T_H, 501)
ax.fill_between(t, np.maximum(PLAN_LINE * t - 2 * np.sqrt(RISK_LINE * t), 0), PLAN_LINE * t + 2 * np.sqrt(RISK_LINE * t),
                color="#B71C1C", alpha=0.12, lw=0, label="risk band: plan line ± 2 sd, sd = √(375 × years)")
ax.plot(t, PLAN_LINE * t, color="#B71C1C", lw=1.8, ls="--", label="plan line: 0.15 × 40 = 6 a year")
for i, col in enumerate(("#1F4E79", "#E65100", "#2E7D32", "#5E35B1", "#616161")):
    ev = times[i][times[i] < T_H]
    steps = np.concatenate([[0.0], np.cumsum(cost_sizes[i][:len(ev)])])
    ax.step(np.concatenate([[0.0], ev, [T_H]]), np.concatenate([steps, [steps[-1]]]), where="post", color=col, lw=1.2,
            alpha=0.9, label="five fresh histories: flat stretches are quiet years" if i == 0 else None)
ax.set(xlabel="years", ylabel="cumulative compliance cost (\\$ million)", xlim=(0, T_H), ylim=(-12, 330),
       title="Event risk: a plan line through quiet years, a widening risk band (seed 26108)")
ax.legend(frameon=False, fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

## Panel 4 — Feel the flaw of averages (Theorem 8.5; Proposition 8.5; Online Appendix 8.B.7)

AXIOM Lab 1.8, step 4. In a linear company the average path obeys the deterministic law exactly (Theorem 8.5(i)):
the mean workforce of Worked Example 4.1, with random attrition restored, follows Table 4.5's roll-up. Bend the
function and the average of the outcomes leaves the outcome computed at average inputs (Theorem 8.5(ii)).
Chapter 5's start-up grows geometrically near zero; with demand noise,
$X_t = X_0\exp\bigl((r - \sigma^2/2)t + \sigma W_t\bigr)$, and the exponential is convex. With $r = 0.9$ and
$\sigma = 0.6$ a year the mean grows by the
factor 2.46 after one year and 6.05 after two, the median only by 2.05 and 4.22, and 62 and then 66 percent of
paths fall below the mean: a plan that is right on average is behind on most paths. The more the function
bends, the wider the gap; with no spread it closes (Proposition 8.3). A concave quantity errs the other way, as
the trough of Worked Example 8.1 will show (Panel 7). And a step with sensitivity $L$ multiplies the mean-square
spread it inherits by up to $L^2$ (Proposition 8.5): three expansive steps at 1.2 can triple it, and three
consolidating steps at 0.8 cut it to at most 0.26 of its size.

In [ ]:
# linear: Worked Example 4.1's Unit A (400 legacy, 100 software engineers), a year at a time (Table 4.5)
A4 = np.array([[0.92, 0.0], [0.03, 0.96]])
x = np.array([400.0, 100.0]); EXPECTED = [x.sum()]
for _ in range(5):
    x = A4 @ x; EXPECTED.append(x.sum())
rng = np.random.default_rng(SEED)                      # fresh draws, lab seed: each engineer's year drawn
legacy, soft = np.full(100_000, 400), np.full(100_000, 100); totals = [legacy + soft]
for _ in range(5):
    moves = rng.multinomial(legacy, [0.92, 0.03, 0.05])          # stay legacy, retrain into software, leave
    soft, legacy = rng.binomial(soft, 0.96) + moves[:, 1], moves[:, 0]
    totals.append(legacy + soft)
print("year   Table 4.5 roll-up   mean of 100,000 random workforces (seed 26108)   sd of the fan")
for yr in range(6):
    tt = totals[yr]
    print(f"{yr:4d} {EXPECTED[yr]:15.1f} {tt.mean():25.2f} +/- {tt.std() / sqrt(len(tt)):.2f} {tt.std():24.1f}")

# convex: Online Appendix 8.B.7's start-up
R_G, SIG_G = 0.9, 0.6
GBM = {t: (exp(R_G * t), exp((R_G - SIG_G ** 2 / 2) * t), Phi(SIG_G * sqrt(t) / 2)) for t in (1.0, 2.0)}
z = np.random.default_rng(SEED).standard_normal(200_000)           # fresh draws, lab seed
for t, (mean_f, median_f, below) in GBM.items():
    xs = np.exp((R_G - SIG_G ** 2 / 2) * t + SIG_G * sqrt(t) * z)
    print(f"start-up after {t:.0f} year(s): mean factor {mean_f:.2f}, median {median_f:.2f}, {100 * below:.1f}% of paths below "
          f"the mean (fresh paths: {xs.mean():.2f}, {np.median(xs):.2f}, {100 * (xs < xs.mean()).mean():.1f}%)")
print("bend the function: volatility, mean and median factors after one year, the gap, the share below the mean")
for s in (0.0, 0.2, 0.4, 0.6, 0.8):
    print(f"  sigma {s:.1f}: mean {exp(R_G):.2f}, median {exp(R_G - s * s / 2):.2f}, gap {exp(R_G) - exp(R_G - s * s / 2):.2f}, "
          f"below the mean {100 * Phi(s / 2):.1f}%")
AMPLIFY, DAMP = 1.2 ** 6, 0.8 ** 6
print(f"Proposition 8.5: three steps at L = 1.2 multiply the inherited mean-square spread by up to {AMPLIFY:.2f}; "
      f"three at L = 0.8 by {DAMP:.3f}")

## Panel 5 — Declare and size each channel (Worked Example 8.1, Steps 1–2: \$62.5 million, 2.1 standard deviations, 26 percent)

Worked Example 8.1, Steps 1 and 2. The plan's trough is \$482 million in quarter 7, \$132 million above the floor.
By Theorem 8.3(ii) the deviation's volatility budget is $50/\sqrt{1 - 0.6^2} =$ \$62.5 million, so the trough's
headroom is only **2.1 standard deviations**. By Theorem 8.4, the event channel has a plan line of
$0.10 \times 120 =$ **\$12 million a year** and a $1 - e^{-0.3} =$ **26 percent** chance of at least one closure in
three years. Neither channel alone says how often a path breaches the floor: the deviation has twelve quarters
to find it, and a closure lowers every quarter after it. The mean path is the plan less the closures' plan line
(Theorem 8.4(iii)).

In [ ]:
PL20 = plan(DEPTH_PHASED)
QS = np.arange(QUARTERS + 1)
SD_K = SD_W * np.sqrt((1 - A_E ** (2 * QS)) / (1 - A_E ** 2))         # the deviation's sd by quarter
EXP_LOSS = LAM / 4 * QS * J_MEAN                                       # the closures' plan line, cumulated
MEAN_PATH = pl - EXP_LOSS
print("quarter   plan  phased  headroom  deviation sd  headroom in sd  expected closures  mean path")
for i in QS:
    hs = f"{(pl[i] - FLOOR) / SD_K[i]:14.2f}" if i else f"{'':>14s}"
    print(f"{i:7d} {pl[i]:6.1f} {PL20[i]:7.1f} {pl[i] - FLOOR:9.1f} {SD_K[i]:13.1f} {hs} {EXP_LOSS[i]:18.1f} {MEAN_PATH[i]:10.1f}")
K_LOW, TROUGH, HEAD = int(pl.argmin()), float(pl.min()), float(pl.min() - FLOOR)
BUDGET_E = SD_W / sqrt(1 - A_E ** 2)
print(f"volatility budget 50/sqrt(1 - 0.6^2) = ${BUDGET_E:.1f} million: the trough's headroom, ${HEAD:.0f} million, is "
      f"{HEAD / BUDGET_E:.2f} standard deviations")
P_ONE3 = 1 - exp(-3 * LAM)
print(f"closures: plan line 0.10 x 120 = ${LAM * J_MEAN:.0f} million a year; risk line 0.10 x (120^2 + 50^2) = "
      f"{LAM * (J_MEAN ** 2 + J_SD ** 2):,.0f} a year; at least one in three years 1 - e^-0.3 = {100 * P_ONE3:.1f}%, "
      f"by quarter 7 {100 * (1 - exp(-LAM * 7 / 4)):.1f}%")
print(f"one quarter at the trough, deviation alone: Phi(-{HEAD:.0f}/{BUDGET_E:.1f}) = {100 * Phi(-HEAD / BUDGET_E):.1f}%; "
      f"a path has twelve quarters, and a second channel")
print(f"the mean path bottoms out at ${MEAN_PATH.min():.1f} million in quarter {MEAN_PATH.argmin()}")

## Panel 6 — Simulate (Worked Example 8.1, Step 3; Figure 8.4: 24.6 percent, standard error 0.14 points)

Worked Example 8.1, Step 3, and AXIOM Lab 1.8, step 5. Draw 100,000 three-year quarterly paths: each quarter, add
the persistent deviation and any closures to the plan (Section 8.9). The draws use the book's **random seed 8**
(Volume II, Online Appendix 1.B.4), in the order of the book's engine, so the estimate is the printed one:
liquidity falls below \$350 million in at least one quarter in **24.6 percent** of the paths, with a standard
error of $\sqrt{0.246 \times 0.754/100{,}000} = 0.14$ percentage points. Figure 8.4 draws the fan and the
distribution of each path's lowest quarter.

In [ ]:
L0, CLOSURES0 = simulate()                             # the book's draws: seed 8
P_BOOK, SE_BOOK, LOWS0 = odds(L0)
print(f"the book's draws (seed {BOOK_SEED}), {N_PATHS:,} paths: P(breach) = {100 * P_BOOK:.2f}%, standard error "
      f"sqrt(p(1 - p)/N) = {100 * SE_BOOK:.3f} percentage points")
print(f"95 percent interval {100 * (P_BOOK - 1.96 * SE_BOOK):.1f}% to {100 * (P_BOOK + 1.96 * SE_BOOK):.1f}%: "
      f"five times the board's 5 percent appetite")
LO10, HI90 = np.percentile(L0, [10, 90], axis=0)
print(f"quarter {K_LOW}: 80 percent of paths between ${LO10[K_LOW]:.0f} and ${HI90[K_LOW]:.0f} million around the plan's ${TROUGH:.0f}")

fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6), sharey=True, gridspec_kw={"width_ratios": [2.1, 1]})
for i in range(30):
    ax.plot(QS, L0[i], color="#999999", lw=0.5, alpha=0.6)
ax.fill_between(QS, LO10, HI90, color="#1F4E79", alpha=0.15, lw=0, label="80 percent of paths")
ax.plot(QS, pl, color="#E65100", lw=1.8, ls="--", label=f"plan (trough {TROUGH:.0f})")
ax.plot(QS, L0.mean(axis=0), color="#1F4E79", lw=2, label="mean of all paths")
ax.axhline(FLOOR, color="#B71C1C", lw=1.1, label="board floor 350")
ax.set(xlabel="quarters", ylabel="liquidity (\\$ million)", xlim=(0, 12), ylim=(100, 800), xticks=range(0, 13, 2))
ax.legend(frameon=False, fontsize=7, loc="upper right", ncol=2)
bins = np.arange(50, 630, 10)
cnt, edges = np.histogram(LOWS0, bins=bins)
mid = (edges[1:] + edges[:-1]) / 2
bx.barh(mid, 100 * cnt / N_PATHS, height=9.5, color=["#B71C1C" if m < FLOOR else "#9DB3CC" for m in mid], lw=0)
bx.axhline(FLOOR, color="#B71C1C", lw=1.1)
bx.axhline(TROUGH, color="#E65100", lw=1.0, ls="--", label=f"planned trough {TROUGH:.0f}")
bx.axhline(LOWS0.mean(), color="#1F4E79", lw=1.0, label=f"average trough {LOWS0.mean():.0f}")
bx.plot([], [], "s", color="#B71C1C", label=f"below floor {100 * P_BOOK:.1f}%")
bx.set(xlabel="lowest quarter: % of paths", xlim=(0, 8))
bx.legend(frameon=False, fontsize=7, loc="upper right")
for z in (ax, bx):
    z.tick_params(labelsize=7); z.xaxis.label.set_size(8); z.yaxis.label.set_size(8)
fig.suptitle("One path in four breaches the floor (seed 8, the book's stream)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 7 — Read the answer (Worked Example 8.1, Step 4; Online Appendix 8.B.6: \$384 million, 71 percent, three in four)

Worked Example 8.1, Step 4. The planned trough is not the average trough: averaged over paths, each path's lowest
quarter is **\$384 million**, almost \$100 million below the plan's \$482 million, because the low point of a path
is a concave function of the path and Jensen's inequality works against the plan (Theorem 8.5(ii)). Even the
mean path bottoms out at \$460 million, because the plan line of the event channel accrues \$12 million a year.
The two channels compound: the deviation alone breaches the floor in 8.4 percent of paths and the events alone
in 10.6 percent, but together they do so in 24.6 percent. A path with a closure breaches 71 percent of the time,
and three in four breaches occur on such paths. Online Appendix 8.B.6 adds the decomposition: by total
probability, $0.259 \times 0.709 + 0.741 \times 0.084 = 0.246$, and by the law of total variance (Theorem
8.5(iii)) 40 percent of the uncertainty about a breach is uncertainty about whether a window closes.

In [ ]:
MEAN_SIM = L0.mean(axis=0)
print(f"planned trough ${TROUGH:.1f} million; the mean path's trough ${MEAN_SIM.min():.1f} million (exact {MEAN_PATH.min():.1f}); "
      f"the average of the path troughs ${LOWS0.mean():.1f} million (median {np.median(LOWS0):.1f})")
print(f"Jensen for the concave trough: min of the mean path {MEAN_SIM.min():.1f} >= mean of the minima {LOWS0.mean():.1f}, "
      f"{TROUGH - LOWS0.mean():.0f} below the plan's")
RESULTS = {"Plan as drawn, both channels": (P_BOOK, SE_BOOK, LOWS0, TROUGH)}
for name, kw in (("Execution and market deviation only", {"events": False}), ("Funding-window events only", {"noise": False})):
    p, se, lows = odds(simulate(**kw)[0])
    RESULTS[name] = (p, se, lows, TROUGH)
P_NOISE, P_EVENTS = RESULTS["Execution and market deviation only"][0], RESULTS["Funding-window events only"][0]
print(f"the deviation alone breaches in {100 * P_NOISE:.2f}% of paths, the events alone in {100 * P_EVENTS:.2f}%, "
      f"together in {100 * P_BOOK:.2f}%: they compound")
HIT, BREACH = CLOSURES0 > 0, LOWS0 < FLOOR
PE, P1, P0 = float(HIT.mean()), float(BREACH[HIT].mean()), float(BREACH[~HIT].mean())
SHARE = float((BREACH & HIT).sum() / BREACH.sum())
print(f"{100 * PE:.1f}% of paths suffer a closure and breach {100 * P1:.1f}% of the time, the others {100 * P0:.1f}%; "
      f"{100 * SHARE:.1f}% of breaches are on paths with a closure")
TOTAL = PE * P1 + (1 - PE) * P0
WITHIN = PE * P1 * (1 - P1) + (1 - PE) * P0 * (1 - P0)
BETWEEN = PE * P1 ** 2 + (1 - PE) * P0 ** 2 - TOTAL ** 2
print(f"total probability: {PE:.3f} x {P1:.3f} + {1 - PE:.3f} x {P0:.3f} = {TOTAL:.4f}")
print(f"law of total variance: p(1 - p) = {TOTAL * (1 - TOTAL):.4f} = {WITHIN:.4f} within the two scenarios + {BETWEEN:.4f} "
      f"between them: {100 * BETWEEN / (WITHIN + BETWEEN):.0f}% is whether a window closes")

## Panel 8 — Test the levers (Worked Example 8.1, Step 5; Table 8.5: 12.4, 13.1 and 3.0 percent)

Worked Example 8.1, Step 5, and the rest of AXIOM Lab 1.8, step 5. A committed standby line that replaces up to
\$100 million of any closed funding window cuts the breach probability to **12.4 percent**. Phasing the build so
that its deepest cash draw is \$49 million smaller lifts the trough to \$531 million and cuts the probability to
**13.1 percent**. Together they bring it to **3.0 percent**, within the board's appetite. Every scenario reuses
the book's draws (seed 8), so the levers are compared on the same 100,000 futures (Table 8.5).

In [ ]:
for name, kw in (("Standby line (up to $100 million per closure)", {"line": LINE}), ("Phased build", {"depth": DEPTH_PHASED}),
                 ("Standby line and phased build", {"line": LINE, "depth": DEPTH_PHASED})):
    p, se, lows = odds(simulate(**kw)[0])
    RESULTS[name] = (p, se, lows, float(plan(kw.get("depth", DEPTH)).min()))
print(f"Table 8.5 (seed 8, {N_PATHS:,} paths each)  {'plan trough':>12s} {'breach':>8s} {'s.e.':>6s}  within 5%?")
for name, (p, se, lows, tr) in RESULTS.items():
    print(f"  {name:46s} {tr:9.0f} {100 * p:9.1f}% {100 * se:6.2f}  {'yes' if p <= APPETITE else 'no'}")
DRAW, DRAW20 = pl[0] - TROUGH, PL20[0] - PL20.min()
print(f"phased build: deepest draw ${DRAW20:.0f} million against ${DRAW:.0f} million, ${DRAW - DRAW20:.0f} million smaller; "
      f"trough ${PL20.min():.0f} million in quarter {PL20.argmin()}")
P_LINE = RESULTS["Standby line (up to $100 million per closure)"][0]
P_PHASED, P_BOTH = RESULTS["Phased build"][0], RESULTS["Standby line and phased build"][0]
print(f"either lever alone roughly halves the risk ({P_LINE / P_BOOK:.2f} and {P_PHASED / P_BOOK:.2f} of it); "
      f"only the two together meet the appetite ({100 * P_BOTH:.1f}%)")

## Panel 9 — Vary the number of paths (Section 8.9; Online Appendix 8.B.5: ±4.3, ±1.4 and ±0.4 points)

AXIOM Lab 1.8, step 5, first part. A Monte Carlo estimate is a sample mean, approximately normal around the truth
with standard error $s/\sqrt{N}$, and $\sqrt{p(1-p)/N}$ for a probability: one more decimal digit costs a
hundredfold more paths. Asked of this model, the breach question returns **25.0 ± 4.3** percent with 100 paths,
**25.2 ± 1.4** with 1,000 and **24.5 ± 0.4** with 10,000 (one standard error). The book's engine draws these three
runs from their own streams, seeds $100 + N$ (200, 1,100 and 10,100), and so does the cell below. Pinning a
probability to ±0.1 percentage point takes 47,500 paths near 5 percent and 187,500 near 25 percent (Online
Appendix 8.B.5). A fresh stream drawn with the lab's seed shows the estimate settling as its paths accumulate.

In [ ]:
CONV = []
for n in (100, 1_000, 10_000):
    p, se, _ = odds(simulate(n=n, seed=100 + n)[0])     # the book's engine: each run its own stream, seed 100 + N
    CONV.append((n, p, se))
    print(f"N = {n:7,d} (seed {100 + n:6,d}): {100 * p:5.1f}% +/- {100 * se:.1f}")
print(f"N = {N_PATHS:7,d} (seed      8): {100 * P_BOOK:5.1f}% +/- {100 * SE_BOOK:.2f}")
need = lambda p, h: p * (1 - p) / h ** 2
NEED5, NEED25 = need(0.05, 0.001), need(0.25, 0.001)
print(f"paths to pin a probability to +/-0.1 point: near 5%, {NEED5:,.0f}; near 25%, {NEED25:,.0f}")
print(f"Table 8.5's standard errors run from {100 * min(r[1] for r in RESULTS.values()):.2f} to "
      f"{100 * max(r[1] for r in RESULTS.values()):.2f} points")

fresh, _ = simulate(seed=SEED)                         # a fresh stream with the lab's seed: not the book's draws
b = (fresh[:, 1:].min(axis=1) < FLOOR).astype(float)
nn = np.arange(1, N_PATHS + 1)
run = np.cumsum(b) / nn
band = 2 * np.sqrt(np.maximum(run * (1 - run), 1e-12) / nn)
for n in (100, 1_000, 10_000, 100_000):
    print(f"fresh stream (seed 26108), first {n:7,d} paths: {100 * run[n - 1]:5.2f}% +/- {100 * band[n - 1] / 2:.2f}")
P_FRESH = float(run[-1])
print(f"fresh stream, all paths: {(P_FRESH - P_BOOK) / SE_BOOK:+.1f} standard errors from the book's 24.6%")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.fill_between(nn[49:], 100 * (run - band)[49:], 100 * (run + band)[49:], color="#1F4E79", alpha=0.15, lw=0,
                label="estimate ± 2 standard errors")
ax.plot(nn[49:], 100 * run[49:], color="#1F4E79", lw=1.4, label="estimate from the first N fresh paths")
ax.axhline(100 * P_BOOK, color="#E65100", lw=1.1, ls="--", label=f"the book's 100,000 paths: {100 * P_BOOK:.1f}%")
for n, p, se in CONV:
    ax.errorbar([n], [100 * p], yerr=[100 * se], fmt="o", color="#B71C1C", ms=4, capsize=3, lw=1)
ax.plot([], [], "o", color="#B71C1C", ms=4, label="Section 8.9's runs (seeds 100 + N), ± 1 s.e.")
ax.set_xscale("log")
ax.set_xticks([100, 1_000, 10_000, 100_000], ["100", "1,000", "10,000", "100,000"])
ax.set(xlabel="paths simulated, N", ylabel="breach probability (%)", xlim=(50, N_PATHS), ylim=(10, 40),
       title="The error bar shrinks like one over the square root of the paths (seed 26108)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

## Validation — the book's numbers, and agreement with `DCT_V1_Ch08_Lab.xlsx`

Each checkpoint is a number printed in Chapter 8 (Worked Example 8.1, Table 8.5, Sections 8.6, 8.7 and 8.9,
Figures 8.1 and 8.3), in Online Appendix 8.B, or in Table 4.5, which Section 8.8 cites. Simulated checkpoints
use the book's streams (seed 8; seeds $100 + N$ for Section 8.9's runs). The workbook recomputes the ones marked
*workbook* with Excel formulas and shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
R_ = RESULTS
checks = [
    ("revenue: long-run variance (workbook)", VAR_INF, 441, 0.5),
    ("revenue: volatility budget, $ million (workbook)", BUDGET, 21.0, 0.05),
    ("revenue: inflation from persistence (workbook)", BUDGET / SD_R, 1.40, 0.005),
    ("revenue: $45m shortfall, % a quarter (workbook)", 100 * P_COV, 1.6, 0.05),
    ("revenue: same, persistence ignored, % (workbook)", 100 * P_NAIVE, 0.13, 0.005),
    ("revenue after 1 quarter: mean (workbook)", revenue_law(1)[0], 21, 0.5),
    ("revenue after 1 quarter: sd (workbook)", revenue_law(1)[1], 15, 0.5),
    ("revenue after 2 quarters: mean (workbook)", revenue_law(2)[0], 15, 0.5),
    ("revenue after 2 quarters: sd (workbook)", revenue_law(2)[1], 18, 0.5),
    ("revenue after 4 quarters: mean (workbook)", revenue_law(4)[0], 7, 0.5),
    ("revenue after 4 quarters: sd (workbook)", revenue_law(4)[1], 20, 0.5),
    ("halved persistence: budget, $ million (workbook)", HALF, 16.0, 0.05),
    ("halved persistence: shortfall, % (workbook)", 100 * P_HALF, 0.25, 0.005),
    ("two readings: sigma_11 (workbook)", SIG[0, 0], 0.11605, 5e-6),
    ("two readings: sigma_12 (workbook)", SIG[0, 1], 0.003606, 5e-7),
    ("two readings: sigma_22 (workbook)", SIG[1, 1], 0.015625, 5e-7),
    ("two readings: first sd (workbook)", sqrt(SIG[0, 0]), 0.341, 5e-4),
    ("chain: P^2 on to on (workbook)", P2[0, 0], 0.84, 1e-9),
    ("chain: P^2 off to on (workbook)", P2[1, 0], 0.48, 1e-9),
    ("chain: P^3 on to off, three quarters (workbook)", OFF3, 0.196, 5e-4),
    ("chain: P^3 off to on (workbook)", P3[1, 0], 0.588, 5e-4),
    ("chain: long-run share on plan (workbook)", PI[0], 0.75, 1e-9),
    ("chain: off-plan spell, quarters (workbook)", 1 / P[1, 0], 3.3, 0.05),
    ("regulatory: E[J^2] (workbook)", EJ2, 2500, 1e-9),
    ("regulatory: plan line, $m a year (workbook)", PLAN_LINE, 6, 1e-9),
    ("regulatory: risk line, variance a year (workbook)", RISK_LINE, 375, 1e-9),
    ("regulatory: sd over one year (workbook)", SD1, 19, 0.5),
    ("regulatory: sd over five years (workbook)", SD5, 43, 0.5),
    ("regulatory: mean wait, years (workbook)", 1 / LAM_R, 6.7, 0.05),
    ("regulatory: next year after quiet years, % (workbook)", 100 * P_NEXT, 14, 0.5),
    ("half the frequency, twice the size: variance (workbook)", LAM_R / 2 * 4 * EJ2, 2 * 375, 1e-9),
    ("Table 4.5: Unit A expected total, year 5", EXPECTED[5], 392.1, 0.05),
    ("start-up: mean factor, one year (workbook)", GBM[1.0][0], 2.46, 0.005),
    ("start-up: mean factor, two years (workbook)", GBM[2.0][0], 6.05, 0.005),
    ("start-up: median factor, one year (workbook)", GBM[1.0][1], 2.05, 0.005),
    ("start-up: median factor, two years (workbook)", GBM[2.0][1], 4.22, 0.005),
    ("start-up: below the mean, one year, % (workbook)", 100 * GBM[1.0][2], 62, 0.5),
    ("start-up: below the mean, two years, % (workbook)", 100 * GBM[2.0][2], 66, 0.5),
    ("three steps at 1.2: spread factor (workbook)", AMPLIFY, 3, 0.05),
    ("three steps at 0.8: spread factor (workbook)", DAMP, 0.26, 0.005),
    ("plan: liquidity today, $ million (workbook)", pl[0], 620, 0.5),
    ("plan: trough, $ million (workbook)", TROUGH, 482, 0.5),
    ("plan: quarter of the trough (workbook)", K_LOW, 7, 0),
    ("plan: end of year 3, $ million (workbook)", pl[-1], 510, 0.5),
    ("plan: headroom above the floor (workbook)", HEAD, 132, 0.5),
    ("deviation: volatility budget (workbook)", BUDGET_E, 62.5, 0.05),
    ("deviation: headroom in sd (workbook)", HEAD / BUDGET_E, 2.1, 0.05),
    ("closures: plan line, $m a year (workbook)", LAM * J_MEAN, 12, 1e-9),
    ("closures: one or more in three years, % (workbook)", 100 * P_ONE3, 26, 0.5),
    ("breach probability, % (seed 8)", 100 * P_BOOK, 24.6, 0.05),
    ("its standard error, points (workbook)", 100 * SE_BOOK, 0.14, 0.005),
    ("average trough, $ million (seed 8)", LOWS0.mean(), 384, 0.5),
    ("mean path's trough, simulated (seed 8)", MEAN_SIM.min(), 460, 0.5),
    ("mean path's trough, exact (workbook)", MEAN_PATH.min(), 460, 0.5),
    ("deviation only, % (seed 8)", 100 * P_NOISE, 8.4, 0.05),
    ("events only, % (seed 8)", 100 * P_EVENTS, 10.6, 0.05),
    ("paths with a closure, % (seed 8)", 100 * PE, 25.9, 0.05),
    ("breach given a closure, % (seed 8)", 100 * P1, 70.9, 0.05),
    ("breach given no closure, % (seed 8)", 100 * P0, 8.4, 0.05),
    ("breaches with a closure, %: three in four (seed 8)", 100 * SHARE, 75, 0.5),
    ("total probability (workbook)", TOTAL, 0.246, 5e-4),
    ("breach-indicator variance (workbook)", TOTAL * (1 - TOTAL), 0.186, 5e-4),
    ("variance within scenarios (workbook)", WITHIN, 0.111, 5e-4),
    ("variance between scenarios (workbook)", BETWEEN, 0.075, 5e-4),
    ("between share, % (workbook)", 100 * BETWEEN / (WITHIN + BETWEEN), 40, 0.5),
    ("standby line, % (seed 8)", 100 * P_LINE, 12.4, 0.05),
    ("phased build, % (seed 8)", 100 * P_PHASED, 13.1, 0.05),
    ("standby line and phased build, % (seed 8)", 100 * P_BOTH, 3.0, 0.05),
    ("phased build: plan trough (workbook)", PL20.min(), 531, 0.5),
    ("phased build: draw smaller by (workbook)", DRAW - DRAW20, 49, 0.5),
    ("Table 8.5: smallest standard error (workbook)", 100 * min(r[1] for r in R_.values()), 0.05, 0.005),
    ("100 paths: estimate, % (seed 200)", 100 * CONV[0][1], 25.0, 0.05),
    ("100 paths: standard error (workbook)", 100 * CONV[0][2], 4.3, 0.05),
    ("1,000 paths: estimate, % (seed 1,100)", 100 * CONV[1][1], 25.2, 0.05),
    ("1,000 paths: standard error (workbook)", 100 * CONV[1][2], 1.4, 0.05),
    ("10,000 paths: estimate, % (seed 10,100)", 100 * CONV[2][1], 24.5, 0.05),
    ("10,000 paths: standard error (workbook)", 100 * CONV[2][2], 0.4, 0.05),
    ("paths for 0.1 point near 5% (workbook)", NEED5, 47500, 0.5),
    ("paths for 0.1 point near 25% (workbook)", NEED25, 187500, 0.5),
]
ok, width = 0, max(len(name) for name, *_ in checks)
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol + 1e-9 else "FAIL"; ok += status == "PASS"
    f = ".6f" if abs(want) < 1 else ".4f"
    print(f"{name:{width}s} {got:14{f}}   book {want:12{f}}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 9 turns the company model into an architecture: components with their own states, wired by
flows of demand, cash, capability and policy, and bound by constraints that reach across the wiring; the risk
architecture of Chapter 12 attaches this chapter's distributions to that wiring. AXIOM-08 runs the same model
interactively (Lab 1.8); this notebook and the workbook are its reproducible record.